# GPTPet v2 research scratchpad

Started 2026-09-18. Purpose: collect papers, projects and software that should shape the v2 brain, which will drive the robot only through MCP tools (ai2thor-mcp in sim, gpt-pet-mcp on the real robot).

Constraints that filter everything below:
- 4-wheeled (mecanum) robot, Kinect 360 RGB-D, RPLIDAR, IMU, proximity sensors. No arms, no legs.
- Jetson Nano onboard is weak; SLAM/nav2 already run on the server. The brain will run on the server and/or cloud.
- Latency is the primary design constraint. v1 spent up to ~10 s per loop with the robot standing still.
- Voice interaction is a planned addition.
- Agent framework should be general because all actuation is behind MCP. Google ADK is the current candidate.
- Interest in a System 1 / System 2 split and in ACT-R.

Reading checklist: sections 1-10 use task-list checkboxes. Change `[ ]` to `[x]` in the cell source as you finish each item.


## 0. What v1 (gpt-pet) taught us

Loop: sensory modules -> subconscious input modules (vision LLM + vector-DB view cache, audio, proximity) -> conscious module (task + goal) -> executor (skill library, else code-writing agent -> AST check -> mock -> real) -> report result -> sleep(5).

Where the ~10 s went (all serial, one thread):
1. Vision: a gpt-4o call per iteration unless the image-similarity cache hit (Weaviate img2vec + distance threshold).
2. Conscious: gpt-4o call, plus gpt-4o-mini summarizer for the observation, plus an OutputFixingParser call on bad YAML, plus time-weighted memory retrieval (OpenAI embeddings round trip).
3. Executor: vector search for a matching skill, a 20% random validation LLM call, or a JSON agent loop of up to 5 iterations that writes Python, mock-executes, then really executes.
4. Sim-only overhead: proximity sensors emulated by trial moves and teleport-back in four directions after every step.
5. sleep(5) after each loop so the robot is still for the next frame.

Keep from v1:
- The sensory / subconscious / conscious / executor decomposition, and the goal vs task distinction.
- Skill library with validation and deletion on failure (Voyager pattern).
- "Seen before" novelty signal from image embedding similarity.
- Passageway detection from depth with visual markers for the VLM. ai2thor-mcp now does this better (numbered markers, stable ids, navmesh validation).
- Safety gating before anything touches motors (AST check -> mock -> real).
- Generative-agent memory (recency x importance x relevance + reflection).
- Rollback of the last actions after a failure.

Reconsider in v2:
- Code as the action space. With ~10 primitives on a wheeled robot it cost an agent loop per task and needed the secret-passphrase hack to prove the tool was called. Tool calls against the MCP server remove this entirely (set_nav_goal / do_move / do_rotate).
- One serial thread. Nothing overlapped; every LLM call stopped the robot.
- Six Weaviate collections (views, goals, tasks, skills, objects, memories), each needing an embedding call.
- The conscious prompt did four jobs at once (mentor, next task, goal, goal-completed flag).
- Views were cached per goal_id, so the same view was re-described whenever the goal changed.
- Prompts assumed OpenAI YAML/JSON agent formats from LangChain 0.1.

What ai2thor-mcp already gives v2 (and gpt-pet-mcp must match 1:1):
- get_current_view: objects + passages with stable ids and a numbered image (set-of-marks).
- set_nav_goal / get_nav_status / cancel_nav: nav2 NavigateToPose semantics (async goal, polling, cancel, preemption).
- list_destinations (memory of everything seen), get_map, do_move / do_rotate (preempting teleop), overhead recording.

So the v2 brain's job reduces to: decide where to go and what to do, keep deciding while the robot moves, remember, and talk.


## 1. Fast/slow architectures (the core design question)

Language-agent level (closest to what v2 needs):

- [ ] **Agents Thinking Fast and Slow: A Talker-Reasoner Architecture** (Google DeepMind, 2024). [arXiv 2410.08328](https://arxiv.org/abs/2410.08328). Talker (System 1) is always on; Reasoner (System 2) runs asynchronously and is the only writer of a structured belief state in shared memory; the Talker reads the latest belief and may act on a stale one; it waits only in phases that need planning or tools. Failure modes: snap-judgment Talker, stale beliefs. Open question: deciding automatically when System 2 is needed. Template for v2's S1/S2 contract.
- [x] **Thinking Fast and Slow in AI** (Booch, Rossi et al., AAAI 2021). [arXiv 2010.06002](https://arxiv.org/abs/2010.06002). The original proposal; introduces the metacognitive arbiter idea.
- [x] **SOFAI project site** (IBM). [sites.google.com/view/sofai](https://sites.google.com/view/sofai/home). Overview of the Slow and Fast AI architecture and its papers.
- [x] **Thinking Fast and Slow in Human and Machine Intelligence** (CACM 2025). [ACM](https://dl.acm.org/doi/10.1145/3715709). Current statement of the SOFAI position. Key idea: a metacognitive module arbitrates between S1 (experience-based) and S2 (deliberative) solvers using confidence and cost. The arbitration policy is the real design problem.
- [ ] **A Neurosymbolic Fast and Slow Architecture for Graph Coloring** (SOFAI-v2, 2024). [arXiv 2412.01752](https://arxiv.org/html/2412.01752). Concrete arbitration mechanics in the newest SOFAI version.
- [ ] **Combining Fast and Slow Thinking for Human-like and Efficient Navigation in Constrained Environments** (2022). [arXiv 2201.07050](https://arxiv.org/pdf/2201.07050). SOFAI applied to navigation on a grid; the closest SOFAI paper to a moving robot.
- [ ] **Hi Robot: Open-Ended Instruction Following with Hierarchical Vision-Language-Action Models** (Physical Intelligence, ICML 2025). [arXiv 2502.19417](https://arxiv.org/abs/2502.19417). High-level VLM reasoner emits language subgoals and absorbs situated feedback ("that's not trash"); a low-level policy executes. Same S2-emits-subgoal, S1-executes shape, with language as the interface.

Runtime and systems papers on planning while acting:

- [ ] **TypeGo: An OS Runtime for Embodied Agents** (Jul 2026). [arXiv 2607.05482](https://arxiv.org/abs/2607.05482). Asynchronous LLM loops at multiple timescales overlapping execution; a "Skill Kernel" arbitrates typed physical subsystems among per-task processes with preempt/resume/replace; speculative skill streaming hides LLM latency; a fast first-action path gives visible feedback within a second. Unitree Go2 prototype: 50% lower per-step delay, 73% better time-to-first-action.
- [ ] **AgenticCache: Cache-Driven Asynchronous Planning for Embodied AI Agents** (Apr 2026). [arXiv 2604.24039](https://arxiv.org/abs/2604.24039). Exploits plan locality: a cache of plan transitions is served immediately while a background updater validates with the LLM. Reported 65% lower latency, 50% fewer tokens.
- [ ] **Parallelized Planning-Acting for Efficient LLM-based Multi-Agent Systems** (AAMAS 2026). [arXiv 2503.03505](https://arxiv.org/abs/2503.03505). Dual-thread planning and acting with interruptible execution.
- [ ] **Agentic AI for Robot Control: Flexible but still Fragile** (Feb 2026). [arXiv 2602.13081](https://arxiv.org/abs/2602.13081). Real deployments on two robots; explicit event checks and operator interventions that redirect execution; notes that agent frameworks lack hard preemption of tool calls, so robot skills must be cancellable and monitored concurrently. Candid about nondeterminism and prompt sensitivity.
- [ ] **Robotouille: An Asynchronous Planning Benchmark for LLM Agents** (ICLR 2025). [alphaXiv 2502.05227](https://www.alphaxiv.org/abs/2502.05227v1). Benchmark for agents that must plan over overlapping actions and interruptions.

VLA-level dual systems (manipulation-focused, but the pattern transfers):

- [ ] **Helix: A Vision-Language-Action Model for Generalist Humanoid Control** (Figure). [figure.ai/news/helix](https://www.figure.ai/news/helix). S2 VLM at 7-9 Hz, S1 policy at 200 Hz; Helix 02 added a System 0.
- [ ] **GR00T N1: An Open Foundation Model for Generalist Humanoid Robots** (NVIDIA, 2025). [arXiv 2503.14734](https://arxiv.org/html/2503.14734v1). S2 VLM at 10 Hz, S1 diffusion transformer at 120 Hz.
- [ ] **Hume: Introducing System-2 Thinking in Vision-Language-Action Model** (2025). [arXiv 2505.21432](https://arxiv.org/html/2505.21432v1).
- [ ] **A Dual Process VLA: Efficient Robotic Manipulation Leveraging VLM** (2024). [arXiv 2410.15549](https://arxiv.org/pdf/2410.15549).
- [ ] **Real-Time Execution of Action Chunking Flow Policies** (Physical Intelligence). [paper](https://www.pi.website/download/real_time_chunking.pdf). Decouple prediction from execution: keep executing the current chunk while the next is computed, and blend at the seam.
- [ ] **LeRobot docs: Asynchronous Inference**. [huggingface.co/docs/lerobot/async](https://huggingface.co/docs/lerobot/async). Practical write-up of the decoupled policy-server pattern.
- [ ] **LeRobot docs: Real-Time Chunking**. [huggingface.co/docs/lerobot/rtc](https://huggingface.co/docs/lerobot/rtc). GPTPet analogue: keep driving the current nav goal while S2 chooses the next; the "chunk" is a nav goal plus a fallback behavior.

Classic layered control (still the right vocabulary):

- [ ] **Subsumption architecture** (Brooks, 1986). [Wikipedia](https://en.wikipedia.org/wiki/Subsumption_architecture).
- [ ] **Three-layer architecture** (Gat / Firby). [Wikipedia](https://en.wikipedia.org/wiki/Three-layer_architecture). Deliberator, sequencer, reactive skills. Map: S2 = deliberator, S1 = sequencer, nav2 = skills.
- [ ] **BehaviorTree.CPP docs** (the BT engine nav2 uses). [behaviortree.dev](https://www.behaviortree.dev/). Behavior trees as the sequencer layer.
- [ ] **BTGenBot: Behavior Tree Generation for Robotic Tasks with Lightweight LLMs** (IROS 2024). [arXiv 2403.12761](https://arxiv.org/abs/2403.12761), [code](https://github.com/AIRLab-POLIMI/BTGenBot). Small LLMs emit BT XML executed by BehaviorTree.CPP with nav2 MoveTo nodes.
- [ ] **BTGenBot-2: Efficient Behavior Tree Generation with Small Language Models** (2026). [arXiv 2602.01870](https://arxiv.org/html/2602.01870).
- [ ] **LLM-as-BT-Planner** (2024). [arXiv 2409.10444](https://arxiv.org/abs/2409.10444).
- [ ] **Code-BT: A Code-Driven Approach to Behavior Tree Generation for Robot Tasks** (IJCAI 2025). [paper](https://www.ijcai.org/proceedings/2025/0980.pdf).

Working notes for the v2 design (not readings; to be validated):

- S1: an always-on loop at the rate of get_current_view. Picks among current destinations with cheap scoring (novelty, drives, last heard command), reacts to nav failure / proximity / speech, and never waits on S2. Candidate implementations: no LLM (rules + scores), a small local model, or a Flash-class model with a strict latency budget.
- S2: background reasoner that writes a belief + intent document (goal, plan, interesting things, people) to shared memory. Triggered by events: unseen view, person or animal detected, speech, nav failure, goal completed, idle timer. Never blocks S1; S1 reads the latest belief (Talker-Reasoner).
- Arbitration: explicit, inspectable, and measured (SOFAI metacognition). Preemption goes through cancel_nav.


## 2. Cognitive architectures: CoALA, ACT-R, Soar, Global Workspace Theory

Frameworks and vocabulary:

- [ ] **Cognitive Architectures for Language Agents (CoALA)** (2023). [arXiv 2309.02427](https://arxiv.org/abs/2309.02427). Working vs long-term memory (episodic, semantic, procedural), internal vs external actions, a decision loop with planning and execution. Use its vocabulary to write the v2 design doc.
- [ ] **awesome-language-agents** (companion list to CoALA). [GitHub](https://github.com/ysymyth/awesome-language-agents).
- [ ] **A Review of 40 Years of Cognitive Architecture Research** (2016). [arXiv 1610.08602](https://arxiv.org/pdf/1610.08602). Survey of the field.

ACT-R:

- [ ] **ACT-R official software and tutorials** (CMU; Lisp, with Docker + Jupyter). [act-r.psy.cmu.edu/software](https://act-r.psy.cmu.edu/software/). Do the first tutorial units to learn buffers, productions, and activation.
- [ ] **pyactr** (Python ACT-R). [GitHub](https://github.com/jakdot/pyactr).
- [ ] **python_actr** (Carleton). [GitHub](https://github.com/CarletonCognitiveModelingLab/python_actr).
- [ ] **gactar** (one model file, several ACT-R implementations). [GitHub](https://github.com/asmaloney/gactar).
- [ ] **ACT-R/E: An Embodied Cognitive Architecture for Human-Robot Interaction** (Trafton et al., NRL, JHRI 2013). [ACM](https://dl.acm.org/doi/10.5898/JHRI.2.1.Trafton). ACT-R on robots; adds configural (navigation) and manipulative space modules.
- [ ] **Human-Like Remembering and Forgetting in LLM Agents: An ACT-R-Inspired Memory Architecture** (HAI 2025). [ACM](https://dl.acm.org/doi/10.1145/3765766.3765803). ACT-R activation as the retrieval and forgetting rule for an LLM agent.
- [ ] **Cognitive LLMs: Integrating Cognitive Architectures and LLMs (LLM-ACTR)** (2024). [arXiv 2408.09176](https://arxiv.org/pdf/2408.09176).
- [ ] **Synergistic Integration of LLMs and Cognitive Architectures for Robust AI** (AAAI Symposium 2024). [AAAI](https://ojs.aaai.org/index.php/AAAI-SS/article/view/27706).
- [ ] **FadeMem: Biologically-Inspired Forgetting for Efficient Agent Memory** (2026). [arXiv 2601.18642](https://arxiv.org/html/2601.18642). Adaptive decay by relevance, access frequency, and time.
- [ ] **Human-Inspired Memory Architecture for LLM Agents** (2026). [arXiv 2605.08538](https://arxiv.org/abs/2605.08538). Consolidation, interference-based forgetting, reconsolidation on retrieval.
- [ ] **From Human Memory to AI Memory: A Survey on Memory Mechanisms in the Era of LLMs** (2025). [arXiv 2504.15965](https://arxiv.org/html/2504.15965v2).
- [ ] **NL2GenSym: Natural Language to Generative Symbolic Rules for Soar via LLMs** (2025). [arXiv 2510.09355](https://arxiv.org/pdf/2510.09355). The Soar-side equivalent.

Global Workspace Theory (what v1's conscious/subconscious naming was reaching for):

- [ ] **LIDA: A Computational Model of Global Workspace Theory and Developmental Learning** (Franklin et al.). [AAAI](https://aaai.org/papers/0011-fs07-01-011-%EF%80%A0lida-a-computational-model-of-global-workspace-theory-and-developmental-learning/), [Wikipedia](https://en.wikipedia.org/wiki/LIDA_(cognitive_architecture)).
- [ ] **CTM-AI: A Blueprint for General AI Inspired by a Model of Consciousness** (2026). [arXiv 2605.04097](https://arxiv.org/pdf/2605.04097). First practical instantiation of Blum's Conscious Turing Machine.
- [ ] **"Theater of Mind" for LLMs: A Cognitive Architecture Based on Global Workspace Theory** (Apr 2026). [arXiv 2604.08206](https://arxiv.org/abs/2604.08206). Event-driven broadcast hub, specialist agents compete for the workspace, entropy-based intrinsic drive.
- [ ] **Unified Mind Model: Reimagining Autonomous Agents in the LLM Era** (2025). [arXiv 2503.03459](https://arxiv.org/abs/2503.03459). GWT as macro-architecture; MindOS engine.

Drives and affect for a pet:

- [ ] **Kismet's homeostatic drives** (Breazeal, Designing Sociable Robots, MIT Press 2002). Needs as drives with a homeostatic band; behavior selection restores the band.
- [ ] **Emotion in Reinforcement Learning Agents and Robots: A Survey** (2017). [arXiv 1705.05172](https://arxiv.org/pdf/1705.05172).
- [ ] **Human-Robot Mutual Learning through Affective-Linguistic Interaction and Differential Outcomes Training** (2024). [arXiv 2407.01280](https://arxiv.org/pdf/2407.01280). Reachy with homeostatically controlled needs driving its motivational state.
- [ ] **Design and Implementation of a Companion Robot with LLM-Based Hierarchical Emotion Motion Generation** (Applied Sciences 2025). [DOI](https://doi.org/10.3390/app152312759).
- [ ] **A motivational-based learning model for mobile robots** (2024). [ScienceDirect](https://www.sciencedirect.com/science/article/abs/pii/S138904172400072X).

Notes (not readings):

- What ACT-R can contribute to v2: (a) declarative memory activation = base-level learning (recency and frequency) + spreading activation + partial matching, a principled retrieval and forgetting rule instead of raw cosine similarity; (b) productions with utility learning as a cheap S1 policy; (c) modules and buffers as a bounded working memory; (d) explicit timing.
- GWT gives a principled answer to "what wakes System 2": modules compete to broadcast, and a salient broadcast (novel object, heard command, stuck) recruits deliberation.
- Idea: needs (energy, social contact, curiosity, play) as homeostatic variables that bias S1 scoring and S2 goals.


## 3. Memory for an embodied pet

- [ ] **Generative Agents: Interactive Simulacra of Human Behavior** (Park et al., 2023). [arXiv 2304.03442](https://arxiv.org/abs/2304.03442). v1 used LangChain's GenerativeAgentMemory; keep the recency x importance x relevance retrieval and reflection, drop the langchain_experimental dependency.
- [ ] **ReMEmbR: Building and Reasoning Over Long-Horizon Spatio-Temporal Memory for Robot Navigation** (NVIDIA, 2024). [arXiv 2409.13682](https://arxiv.org/pdf/2409.13682). VLM captions of short video segments with timestamp and pose go into a vector DB; an LLM agent queries iteratively; deployed on a wheeled Nova Carter on Jetson. Very close to "pet memory of where things and people are".
- [ ] **ReMEmbR code**. [GitHub](https://github.com/NVIDIA-AI-IOT/remembr). Read the memory builder and the query agent.
- [ ] **ReMEmbR NVIDIA technical blog**. [blog](https://developer.nvidia.com/blog/using-generative-ai-to-enable-robots-to-reason-and-act-with-remembr/).
- [ ] **Embodied-RAG: General Non-parametric Embodied Memory for Retrieval and Generation** (2024). [arXiv 2409.18313](https://arxiv.org/html/2409.18313v4). Hierarchical semantic-forest memory over robot experience for navigation and QA.
- [ ] **ConceptGraphs: Open-Vocabulary 3D Scene Graphs for Perception and Planning** (2023). [arXiv 2309.16650](https://arxiv.org/abs/2309.16650). Open-vocab 3D scene graph from posed RGB-D.
- [ ] **Clio: Real-time Task-Driven Open-Set 3D Scene Graphs** (MIT SPARK, 2024). [arXiv 2404.13696](https://arxiv.org/abs/2404.13696), [code](https://github.com/MIT-SPARK/Clio). Information-bottleneck clustering keeps only task-relevant objects and places.
- [ ] **Hydra: real-time 3D scene graph construction** (MIT SPARK). [GitHub](https://github.com/MIT-SPARK/Hydra). The places layer Clio builds on.
- [ ] **EmbodiedLGR: Lightweight Graph Representation and Retrieval for Semantic-Spatial Memory** (2026). [arXiv 2604.18271](https://arxiv.org/html/2604.18271v1).
- [ ] **A Scene Graph Backed Approach to Open Set Semantic Mapping** (2026). [arXiv 2602.03781](https://arxiv.org/pdf/2602.03781).
- [ ] **LLM-Empowered Embodied Agent for Memory-Augmented Task Planning in Household Robotics** (2025). [arXiv 2504.21716](https://arxiv.org/abs/2504.21716).
- [ ] **Mem0 vs Letta (MemGPT): AI Agent Memory Compared** (2026). [vectorize.io](https://vectorize.io/articles/mem0-vs-letta). Drop-in memory layer vs agent runtime with core / archival tiers.
- [ ] **Best AI Agent Memory Frameworks in 2026**. [atlan.com](https://atlan.com/know/best-ai-agent-memory-frameworks-2026/).

Notes (not readings):

- With Kinect RGB-D and nav2 poses a scene graph is feasible on the server GPU.
- v1's six Weaviate collections plus an embedding call each are part of the latency; a single store with ACT-R-style activation on top is worth prototyping.


## 4. Perception and navigation for a camera-only wheeled robot

Marker prompting (ai2thor-mcp's get_current_view already implements this):

- [ ] **Set-of-Mark Prompting Unleashes Extraordinary Visual Grounding in GPT-4V** (2023). [arXiv 2310.11441](https://arxiv.org/abs/2310.11441).
- [ ] **PIVOT: Iterative Visual Prompting Elicits Actionable Knowledge for VLMs** (Google, 2024). [arXiv 2402.07872](https://arxiv.org/abs/2402.07872). Read for the iterative refinement of numbered candidates.
- [ ] **VLMnav: End-to-End Navigation with Vision Language Models: Transforming Spatial Reasoning into Question-Answering** (2024). [arXiv 2411.05755](https://arxiv.org/abs/2411.05755). Read for prompt design and the ablations.

Zero-shot exploration and object-goal navigation with LLM/VLM-scored frontiers:

- [ ] **VLFM: Vision-Language Frontier Maps for Zero-Shot Semantic Navigation** (2023). [arXiv 2312.03275](https://www.emergentmind.com/papers/2312.03275). Occupancy frontiers scored by a language-grounded value map.
- [ ] **UniGoal: Towards Universal Zero-shot Goal-oriented Navigation** (CVPR 2025). [paper](https://openaccess.thecvf.com/content/CVPR2025/papers/Yin_UniGoal_Towards_Universal_Zero-shot_Goal-oriented_Navigation_CVPR_2025_paper.pdf).
- [ ] **WMNav: Integrating VLMs into World Models for Object Goal Navigation** (2025). [arXiv 2503.02247](https://arxiv.org/pdf/2503.02247).
- [ ] **PIGEON: VLM-Driven Object Navigation via Points of Interest Selection** (2025). [arXiv 2511.13207](https://arxiv.org/pdf/2511.13207).
- [ ] **DyNaVLM: Zero-Shot VLN with Dynamic Viewpoints and Self-Refining Graph Memory** (2025). [arXiv 2506.15096](https://arxiv.org/html/2506.15096v1).
- [ ] **Think, Remember, Navigate: Zero-Shot Object-Goal Navigation with VLM-Powered Reasoning** (2025). [arXiv 2511.08942](https://arxiv.org/html/2511.08942v1).
- [ ] **History-Augmented VLMs for Frontier-Based Zero-Shot Object Navigation** (2025). [arXiv 2506.16623](https://arxiv.org/pdf/2506.16623).
- [ ] **Stairway to Success: Floor-Aware Zero-Shot Object-Goal Navigation via LLM-Driven Coarse-to-Fine Exploration** (2025). [arXiv 2505.23019](https://arxiv.org/pdf/2505.23019).
- [ ] **LocalNav: Distilling Frontier VLMs and Embodied RL for On-Device Object Goal Navigation** (2026). [arXiv 2606.27871](https://arxiv.org/pdf/2606.27871).

Streaming perception (an S1 that never stops looking):

- [ ] **StreamingVLM: Real-Time Understanding for Infinite Video Streams** (NeurIPS 2025). [arXiv 2510.09608](https://arxiv.org/pdf/2510.09608).
- [ ] **LiveVLM: Efficient Online Video Understanding via Streaming-Oriented KV Cache and Retrieval** (2025). [arXiv 2505.15269](https://arxiv.org/html/2505.15269v2).
- [ ] **Event-VStream: Event-Driven Real-Time Understanding for Long Video Streams** (2026). [arXiv 2601.15655](https://arxiv.org/pdf/2601.15655).
- [ ] **Awesome-Streaming-Video-Understanding** (curated list). [GitHub](https://github.com/sotayang/Awesome-Streaming-Video-Understanding).

Hosted embodied reasoning:

- [ ] **Gemini Robotics ER 2 developer docs**. [ai.google.dev](https://ai.google.dev/gemini-api/docs/robotics-overview). Two endpoints: gemini-robotics-er-2-preview and gemini-robotics-er-2-streaming-preview (Live API, low-latency function calling). Pointing, tracking, boxes, trajectories; task orchestration against custom robot APIs; video moment finding and progress classification; thinking level trades latency.
- [ ] **Gemini Robotics ER 2 announcement** (DeepMind). [deepmind.google](https://deepmind.google/models/gemini-robotics/gemini-robotics-er/).

Edge VLMs, if any perception moves onboard:

- [ ] **Moondream** (0.5B / 2B; runs on Jetson). [moondream.ai](https://moondream.ai/).
- [ ] **SmolVLM** (Hugging Face blog). [blog](https://huggingface.co/blog/smolvlm).
- [ ] **NVIDIA live-vlm-webui: list of VLMs that run on Jetson**. [GitHub](https://github.com/NVIDIA-AI-IOT/live-vlm-webui/blob/main/docs/usage/list-of-vlms.md).
- [ ] **Getting Started with Edge AI on NVIDIA Jetson: LLMs, VLMs, and Foundation Models for Robotics** (NVIDIA blog, 2026). [blog](https://developer.nvidia.com/blog/getting-started-with-edge-ai-on-nvidia-jetson-llms-vlms-and-foundation-models-for-robotics/).
- [ ] **Getting Started with VLM on Jetson Nano** (LearnOpenCV). [article](https://learnopencv.com/vlm-on-jetson-nano/).
- [ ] **A Survey on Efficient Vision-Language Models** (2025). [arXiv 2504.09724](https://arxiv.org/pdf/2504.09724).

Notes (not readings):

- For a pet, "explore" is frontier selection scored by interestingness rather than by a target object.
- Gemma 3 4B QAT fits an Orin Nano. The original Jetson Nano is too small for any of these; the server GPU or an Orin Nano Super is the realistic host.


## 5. Skills, code as action, and curiosity

Skill libraries:

- [ ] **Voyager: An Open-Ended Embodied Agent with Large Language Models** (2023). [arXiv 2305.16291](https://arxiv.org/abs/2305.16291). v1's blueprint: automatic curriculum, skill library, iterative prompting with environment feedback.
- [ ] **ASPIRE: Agentic Skills Discovery for Robotics** (2026). [arXiv 2607.00272](https://arxiv.org/pdf/2607.00272).
- [ ] **Growing with Your Embodied Agent: A Human-in-the-Loop Lifelong Code Generation Framework** (2025). [arXiv 2509.18597](https://arxiv.org/pdf/2509.18597).
- [ ] **Efficient Skill Grounding via Code Refactoring with Small Language Models** (2026). [arXiv 2606.07999](https://arxiv.org/pdf/2606.07999).

Classics:

- [ ] **Code as Policies: Language Model Programs for Embodied Control** (2022). [arXiv 2209.07753](https://arxiv.org/abs/2209.07753).
- [ ] **Inner Monologue: Embodied Reasoning through Planning with Language Models** (2022). [arXiv 2207.05608](https://arxiv.org/abs/2207.05608).
- [ ] **Do As I Can, Not As I Say (SayCan)** (2022). [arXiv 2204.01691](https://arxiv.org/abs/2204.01691).

Code-writing agents (only if v2 keeps any code as action):

- [ ] **smolagents CodeAgent** (Hugging Face). [repo](https://github.com/huggingface/smolagents), [docs](https://huggingface.co/docs/smolagents/en/index). Agent writes Python that calls tools; MCP via ToolCollection.from_mcp with structured outputs. Replaces v1's environment_tool loop outright.

Curiosity:

- [ ] **OMNI: Open-endedness via Models of human Notions of Interestingness** (2023; evaluated in AI2-THOR). [arXiv 2306.01711](https://arxiv.org/html/2306.01711v3). Foundation model as a model of interestingness picks learnable and interesting next tasks.
- [ ] **OMNI-EPIC: Environments Programmed in Code** (2024). [arXiv 2405.15568](https://arxiv.org/abs/2405.15568).
- [ ] **Intrinsically-Motivated Humans and Agents in Open-World Exploration** (2025). [arXiv 2503.23631](https://arxiv.org/pdf/2503.23631).
- [ ] **Interestingness as an Inductive Heuristic for Future Compression Progress** (2026). [arXiv 2605.14831](https://arxiv.org/pdf/2605.14831).

Notes (not readings):

- Default plan is plain tool calls against the MCP server instead of generated code.
- The interestingness model is the pet's curiosity; it can score frontiers and destinations in S2 and cache scores for S1.


## 6. Agent framework and MCP plumbing

Google ADK:

- [ ] **ADK site**. [adk.dev](https://adk.dev/).
- [ ] **ADK MCP tools page**. [adk.dev/tools-custom/mcp-tools](https://adk.dev/tools-custom/mcp-tools/). McpToolset with StdioConnectionParams, SseConnectionParams, StreamableHTTPConnectionParams (ai2thor-mcp already speaks streamable HTTP on /mcp), tool_filter, asyncio lifecycle caveats.
- [ ] **ADK bidi streaming dev guide, part 1**. [adk.dev/streaming/dev-guide/part1](https://adk.dev/streaming/dev-guide/part1/).
- [ ] **ADK bidi streaming dev guide, part 5: audio, images, video**. [guide](https://google.github.io/adk-docs/streaming/dev-guide/part5/).
- [ ] **Codelab: Introduction to ADK Gemini Live API Toolkit**. [codelab](https://codelabs.developers.google.com/intro-to-adk-live).
- [ ] **ADK release notes** (v2.9.0, 2026-09-10: MCP SDK 2.x support, LiveKit runner for voice, YAML graph workflows, model failover). [GitHub releases](https://github.com/google/adk-python/releases).

Framework comparisons:

- [ ] **Comparing Open-Source AI Agent Frameworks** (Langfuse). [blog](https://langfuse.com/blog/2025-03-19-ai-agent-comparison).
- [ ] **AI Agent Frameworks (2026 Update): 8 SDKs Compared** (morphllm). [article](https://www.morphllm.com/ai-agent-framework).
- [ ] **AI Agent Frameworks Compared (2026)** (Agentmelt). [article](https://agentmelt.com/blog/ai-agent-frameworks-compared-2026/).

ROS 2 MCP servers (reference for the real-robot server):

- [ ] **robotmcp/ros-mcp-server** (rosbridge-based; topics, services, actions; ROS 1 and 2). [GitHub](https://github.com/robotmcp/ros-mcp-server).
- [ ] **Yutarop/ros-mcp**. [GitHub](https://github.com/Yutarop/ros-mcp).
- [ ] **kakimochi/ros2-mcp-server** (FastMCP node publishing cmd_vel). [GitHub](https://github.com/kakimochi/ros2-mcp-server).
- [ ] **sandraschi/ros-mcp** (full ROS 2 surface: topics, services, params, bags, launch). [GitHub](https://github.com/sandraschi/ros-mcp).
- [ ] **wise-vision/ros2_mcp**. [GitHub](https://github.com/wise-vision/ros2_mcp).
- [ ] **lpigeon/ros_mcp_server**. [GitHub](https://github.com/lpigeon/ros_mcp_server).

NVIDIA reference for a wheeled Jetson robot with an LLM agent:

- [ ] **ReMEmbR on Nova Carter** (The Robot Report). [article](https://www.therobotreport.com/remembr-generative-ai-enables-robots-reason-act-says-nvidia/).
- [ ] **Isaac ROS**. [developer.nvidia.com/isaac](https://developer.nvidia.com/isaac).

Notes (not readings):

- To check: whether a live (bidi) ADK agent can call MCP tools with the same lifecycle as an LlmAgent, and how long-running tools (nav goals) surface as background work.
- Since tools are MCP, the framework choice is really about async and streaming primitives, interrupt and cancel semantics, and voice support. Alternatives: LangGraph (checkpointing, interrupts), Pydantic AI, OpenAI Agents SDK, Claude Agent SDK, Strands, Microsoft Agent Framework, smolagents.
- Position: do not expose raw ROS to the brain; keep the ai2thor-mcp contract and implement it over rclpy nav2 actions. The ROS MCP repos are useful for transport and lifecycle patterns only.


## 7. Talking to the robot

Hosted and framework options:

- [ ] **Gemini Live API docs**. [ai.google.dev](https://ai.google.dev/gemini-api/docs/live-api). ADK bidi streaming sits on it.
- [ ] **Pipecat: Building with Gemini Live**. [docs](https://docs.pipecat.ai/pipecat/features/gemini-live).
- [ ] **Voice AI Agents on LiveKit: 2026 Engineer's Playbook**. [article](https://www.forasoft.com/blog/article/voice-ai-agents-livekit-guide). Turn detection, noise cancellation; ADK 2.9 ships a LiveKit runner.
- [ ] **Pipecat vs LiveKit Agents**. [article](https://www.dograh.com/feeds/blog/pipecat-livekit-agents).
- [ ] **Voice AI agents in production 2026: LiveKit Agents, OpenAI Realtime, Pipecat, Vapi, Retell compared**. [article](https://www.reactify-solutions.com/articles/voice-ai-agents-production-2026).

Open and local:

- [ ] **huggingface/speech-to-speech** (repo). [GitHub](https://github.com/huggingface/speech-to-speech). Silero VAD -> Parakeet STT -> LLM -> Qwen3-TTS or Kokoro, exposed as an OpenAI Realtime-compatible /v1/realtime WebSocket.
- [ ] **Hugging Face and Cerebras bring Gemma 4 to real-time voice AI** (July 2026). [blog](https://huggingface.co/blog/cerebras-gemma4-voice-ai). The stack powering 9,000+ Reachy Minis.
- [ ] **Reachy Mini goes fully local** (Hugging Face blog). [blog](https://huggingface.co/blog/local-reachy-mini-conversation).
- [ ] **Kyutai: Moshi (full duplex, ~200 ms), Unmute, Kyutai TTS**. [kyutai.org/tts](https://kyutai.org/tts/).
- [ ] **faster-whisper + Silero VAD + Kokoro offline chatbot** (example repo). [GitHub](https://github.com/AliBinary/local-ai-voice-chatbot).

Notes (not readings):

- Speech is an S1 input event that can recruit S2; barge-in maps to cancel_nav; the reply voice should not wait on S2 (Talker-Reasoner).


## 8. Companion and pet robot reference projects

- [ ] **Reachy Mini docs** (Pollen Robotics / Hugging Face). [docs](https://huggingface.co/docs/reachy_mini/en/index). Behaviors published on the Hub, expressive idle motion, VAD -> STT -> LLM -> TTS pipeline. Best current example of an open LLM companion product.
- [ ] **Reachy Mini launch post**. [blog](https://huggingface.co/blog/reachy-mini).
- [ ] **wire-pod** (Anki Vector server). [GitHub](https://github.com/kercre123/wire-pod). Local voice server that routes unmatched utterances to an LLM; Vector's original emotion engine and behavior system are worth studying for pet idle behavior.
- [ ] **pidog-embodiment**. [GitHub](https://github.com/rockywuest/pidog-embodiment). Brain/body split over HTTP, LLM-driven autonomous behaviors, works with any LLM.
- [ ] **Ai-Robo-Dog**. [GitHub](https://github.com/marceld23/Ai-Robo-Dog).
- [ ] **Petoi OpenCat**. [GitHub](https://github.com/PetoiCamp/OpenCat-Quadruped-Robot).
- [ ] **Everything-LLMs-And-Robotics** (big index). [GitHub](https://github.com/jrin771/Everything-LLMs-And-Robotics).


## 9. Evaluation

- [ ] **EmbodiedBench: Comprehensive Benchmarking MLLMs for Vision-Driven Embodied Agents** (ICML 2025 oral). [arXiv 2502.09560](https://arxiv.org/pdf/2502.09560), [code](https://github.com/EmbodiedBench/EmbodiedBench). AI2-THOR navigation and household tasks with Gym-style APIs.
- [ ] **EmbodiedEval: Evaluate Multimodal LLMs as Embodied Agents** (2025). [arXiv 2501.11858](https://arxiv.org/pdf/2501.11858).
- [ ] **Robotouille** (asynchronous planning benchmark; listed in section 1).

Notes (not readings):

- GPTPet-specific metrics to define early: fraction of wall-clock time moving, time-to-first-action after boot, novel views per minute, latency from person detected to reaction, nav success rate, cost per hour.


## 10. Surveys

- [ ] **Large Language Models for Robotics: A Survey** (2023). [arXiv 2311.07226](https://arxiv.org/html/2311.07226v2).
- [ ] **A Survey on Robotics with Foundation Models: toward Embodied AI** (2024). [arXiv 2402.02385](https://arxiv.org/abs/2402.02385).
- [ ] **Foundation Models in Robotics: A Comprehensive Review of Methods, Models, Datasets, Challenges and Future Research Directions** (2026). [arXiv 2604.15395](https://arxiv.org/html/2604.15395v1).
- [ ] **Embodied AI: From LLMs to World Models** (2025). [arXiv 2509.20021](https://arxiv.org/pdf/2509.20021).
- [ ] **Vision-and-Language Navigation Today and Tomorrow: A Survey in the Era of Foundation Models** (2024). [arXiv 2407.07035](https://arxiv.org/pdf/2407.07035).
- [ ] **Vision-Language-Action Models: Concepts, Progress, Applications and Challenges** (2025). [arXiv 2505.04769](https://arxiv.org/pdf/2505.04769).


## 11. Suggested reading order

1. Talker-Reasoner, the SOFAI CACM article, TypeGo, Parallelized Planning-Acting. These settle the fast/slow mechanics and the arbitration question.
2. CoALA, ACT-R/E, the ACT-R-inspired memory paper. Vocabulary and the memory math.
3. ReMEmbR (read the code), PIVOT, VLMnav. Memory and perception that fit the existing MCP tools.
4. ADK MCP docs and streaming guide, Gemini Robotics ER 2 docs. Plumbing.
5. OMNI. Curiosity as the pet's motivation.
6. Reachy Mini docs and huggingface/speech-to-speech. Voice reference implementation.


## 12. Open questions to resolve

- What wakes System 2: fixed events, GWT-style competition, or SOFAI-style metacognition on S1 confidence?
- S1 model choice and latency budget: rules only, a small local model, or a Flash-class model? Measure get_current_view round trip first.
- Belief-state schema shared between S1 and S2, and who is allowed to write which fields.
- Memory store: keep Weaviate, or one store with ACT-R activation on top?
- How the real robot's MCP reports stuck and collision events with the same semantics as the simulator.
- Where the brain runs: server GPU, cloud, or split by system.


## 13. JEPA and latent world models as a System 1 substrate

Added 2026-09-30. This pass was prompted by "jev" — which turned out to mean TypeSafe's **Jev** (§13.1), while the research below first read it as **JEPA** (LeCun's joint-embedding predictive architecture line: I-JEPA, V-JEPA 2, LeJEPA). Happy accident: they are complementary layers, not rivals — JEPA-style encoders watch pixels, Jev judges structured state, Gemini deliberates (§13.1, §14). This section: can JEPA machinery watch every frame cheaply and wake Gemini (System 2) at the right moments? Three blocking questions answered first; design sketch in §14.

**Q1 — Can it run locally with Ollama?** No — and Ollama is the wrong runtime class, not just missing a model:

- Ollama runs GGUF language models on GGML. Its multimodal path is vision-encoder → projector → text decoder, text out only; there is no mode that loads a standalone vision encoder or returns image embeddings, and no JEPA/DINO GGUF graph exists ([Ollama multimodal engine](https://ollama.com/blog/multimodal-models), [llama.cpp vision = mmproj paired with an LLM](https://simonwillison.net/2025/May/10/llama-cpp-vision/)).
- JEPA runs locally fine without it. V-JEPA 2 is MIT-licensed and in HF transformers: `facebook/vjepa2-vitl-fpc64-256` (ViT-L 300M ≈ 0.6 GB fp16; also H 600M, g 1B) via `VJEPA2Model` + `VJEPA2VideoProcessor`, with `skip_predictor=True` for encoder-only. PyTorch MPS handles short clips (community M1 Max report: MPS beat CPU when it fit but failed on 64-frame clips where MLX completed); MLX ports exist ([vjepa2.1-mlx](https://github.com/lukasugar/vjepa2.1-mlx), [vjepa2-mlx](https://github.com/gaarutyunov/vjepa2-mlx)). Frame count is flexible (3D-RoPE; the fpc64 training value "does not impact inference") — 8–16 frames is the latency-sane window. No published ms-level numbers for consumer hardware anywhere; measure here (§14 cell).
- Ollama's only role in this design: an optional local VLM fallback for Gemini outages (`moondream` 1.8b, `qwen3-vl:2b`, `gemma3:4b`) — a captioner that thinks in tokens, i.e. exactly what S1 should not be.

**Q2 — Is it multimodal enough to take GPTPet's frames?** Frames are its *only* modality: input is an RGB clip tensor (B, T, 3, 256, 256), output is patch-token embeddings. It eats camera frames natively but takes no text and emits none — it is not a VLM. Consequences:

- The Gemini→S1 direction channel needs a text-aligned side encoder: SigLIP 2 (Apache 2.0, ViT-B 86M, [arXiv 2502.14786](https://arxiv.org/abs/2502.14786)) embeds Gemini's watch labels ("a person", "an open doorway") once, and S1 scores every frame against them by cosine for microseconds. JEPA/DINO embeddings alone cannot be text-queried. (VL-JEPA, [arXiv 2512.10942](https://arxiv.org/abs/2512.10942), predicts text embeddings — no released weights.)
- The S1→Gemini channel is typed events plus the frame itself: Gemini stays the multimodal interpreter; S1 only decides *when* a frame is worth interpreting.

**Q3 — Is this "automatic thinking all the time"?** Inverted: JEPA doesn't think at all. No tokens, no inner monologue — a fixed-cost forward pass per frame. It is perception, not deliberation, so the Kahneman mapping stays honest: humans never switch perception off either; what is rare is *conscious engagement*. S1 runs always-on but habituated (a view seen twice scores ~0 and produces silence) and holds zero goal authority. The dual-system survey found the same contract in every serious system — Helix, GR00T, Hi Robot, Gemini Robotics-ER: **the slow layer owns the goal; the fast layer executes the current directive with seconds of bounded autonomy and never self-directs**. "The LLM sets direction" is the field consensus, not a constraint to engineer around. Enforcement mechanics in §14.

Readings — the JEPA line:

- [ ] **V-JEPA 2** (Meta, Jun 2025). [arXiv 2506.09985](https://arxiv.org/abs/2506.09985), [repo](https://github.com/facebookresearch/vjepa2), [HF collection](https://huggingface.co/collections/facebook/v-jepa-2-6841bad8413014e185b497a6). ViT-L/H/g encoders trained on >1M h of video; MIT; 77.3% SSv2, 84.0 PerceptionTest. The encoder is the S1 candidate; the predictor is the surprise machine.
- [ ] **V-JEPA 2.1** (Mar 2026, same repo). Adds ViT-B 80M and dense temporally-consistent features @384; torch.hub only — HF transformers support is an [open PR](https://github.com/huggingface/transformers/issues/45496).
- [ ] **V-JEPA 2-AC** (same paper): frozen ViT-g + ~300M block-causal predictor post-trained on <62 h of Droid; plans by CEM minimizing L1 distance to a goal-image embedding; ~16 s/action on an RTX 4090; camera-pose sensitive; 7-D end-effector action space. **Will not zero-shot to discrete AI2-THOR egomotion** — wrong actions, wrong dynamics (arm-under-static-camera vs whole-image motion), wrong latency class. The transferable part is the pattern: frozen encoder + small trained predictor.
- [ ] **Intuitive physics from V-JEPA prediction error** (Feb 2025). [arXiv 2502.11831](https://arxiv.org/abs/2502.11831). Representation-space prediction error as a violation-of-expectation "surprise" signal: >95% pairwise accuracy on IntPhys while pixel-predictive models sit near chance. The surprise signal, validated.
- [ ] **Surprise-gated episodic memory in V-JEPA 2 latent space** (Jun 2026). [arXiv 2606.03787](https://arxiv.org/abs/2606.03787). Bayesian surprise gates which episodes get written to long-term memory — the same gate §14 points at S2 and at memory writes.
- [ ] **LeJEPA** (Balestriero & LeCun, Nov 2025). [arXiv 2511.08544](https://arxiv.org/abs/2511.08544), [code](https://github.com/rbalestr-lab/lejepa). Heuristics-free JEPA objective (SIGReg, ~50 lines, one hyperparameter) — the train-your-own route; no pretrained zoo confirmed.
- [ ] **I-JEPA** (2023, [repo](https://github.com/facebookresearch/ijepa)) / **V-JEPA 1** (2024, [repo](https://github.com/facebookresearch/jepa)): CC BY-NC weights — research only.
- [ ] **AMI Labs** (LeCun post-Meta): [$1.03B seed, Mar 2026](https://techcrunch.com/2026/03/09/yann-lecuns-ami-labs-raises-1-03-billion-to-build-world-models/) to build world models; nothing downloadable as of Sept 2026. Watch for 2027. ("V-JEPA 3" does not exist.)

Readings — latent world models on frozen encoders (the actual S1-for-navigation recipe):

- [ ] **DINO-WM** (2024). [arXiv 2411.04983](https://arxiv.org/abs/2411.04983). Frozen DINOv2 patch features + ~25M trained predictor + CEM: solves unseen mazes and push tasks zero-shot. The blueprint for Stage 1 in §14.
- [ ] **PLDM** (LeCun group, 2025). [arXiv 2502.14819](https://arxiv.org/abs/2502.14819). JEPA latent dynamics for two-room navigation learned from reward-free offline trajectories — i.e. from logs like ours.
- [ ] **JEPA-WMs** (Meta, 2025). [arXiv 2512.24497](https://arxiv.org/abs/2512.24497), [code+weights](https://github.com/facebookresearch/jepa-wms) (CC BY-NC). What makes JEPA world models plan well, benchmarked across manipulation and PointMaze/Wall navigation.
- [ ] **Navigation World Models** (LeCun, CVPR 2025). [arXiv 2412.03572](https://arxiv.org/abs/2412.03572). 1B diffusion egomotion world model — existence proof for image-goal navigation WMs; too heavy for this S1.
- [ ] 2026 continuations: image-goal nav with frozen DINO + trained action-conditioned predictor ([arXiv 2608.09073](https://arxiv.org/abs/2608.09073)); RAE-NWM in dense DINOv2 token space ([arXiv 2603.09241](https://arxiv.org/abs/2603.09241)).

Readings — fast-path encoders:

- [ ] **DINOv3** (Meta, Aug 2025). [arXiv 2508.10104](https://arxiv.org/abs/2508.10104). ViT-S 21M / B 86M distills of a 7B teacher, best-in-class dense features; custom commercial license (DINOv2 stays clean Apache 2.0).
- [ ] **SigLIP 2** (Google, Feb 2025). [arXiv 2502.14786](https://arxiv.org/abs/2502.14786). Apache 2.0, text-aligned, multilingual — the watch-list scorer and the one encoder Stage 0 strictly needs.

Notes (not readings):

- Encoder arithmetic: DINOv3-S/B or SigLIP 2-B on one 256² frame ≈ 256 tokens at 21–86M params → per-frame real-time on Apple Silicon. V-JEPA 2 ViT-L on a 16-frame clip = 2,048 tokens at 300M params → a lower-cadence temporal module, not the per-frame path. V-JEPA's edge is motion structure in the embedding; DINO/SigLIP are per-frame appearance.
- License ladder: V-JEPA 2 (MIT) > DINOv2 / SigLIP 2 (Apache 2.0) > DINOv3 (custom, commercial-allowed) > I-JEPA / V-JEPA 1 / JEPA-WMs weights (NC).
- Interface lesson from the VLA dual systems (Helix's latent vector, GR00T's cross-attention): co-trained latent interfaces are off the table when S2 is a closed cloud model. With Gemini, the S2↔S1 interface must be language/typed data — and that is exactly what Hi Robot, Gemini Robotics-ER and Galaxea G0 ship, so nothing is lost.
- The "novelty signal from image embedding similarity" kept from v1 (§0) is this section, upgraded: same idea, modern encoder, plus a *predictive* term (surprise) that similarity caches cannot express.

## 13.1 Jev and "System One models" (TypeSafe AI) — the actual "jev"

Added 2026-09-30. **Jev** is the first public "System One model": a schema-constrained, non-autoregressive frontier model that returns **typed, calibrated decisions instead of text**. Out of stealth 2026-09-15 (TypeSafe AI, $40M; CEO Diogo Almeida, ex-OpenAI RLHF/ChatGPT lineage). You send a `state` (JSON or string) plus named questions of three types — **Noul** (true/false probability), **Choice** (categorical distribution), **Score** (scaled rating) — and it answers *all of them in one parallel pass* with probabilities and a calibrated confidence per answer, in 70–500 ms, at $0.042/MTok input, output free. Trained with "Reinforcement Learning for Calibrated Decisions" (RLCD). Flagship demo: playing Doom from structured game state — real-time control decisions, which is the GPTPet shape.

The three blocking questions, for Jev:

- **Local / Ollama?** No and no. API early access ([typesafe.ai](https://typesafe.ai/blog/introducing-system-one-models-and-jev)) or hosted on [Cloudflare Workers AI](https://developers.cloudflare.com/ai/models/typesafe/jev/) (`typesafe/jev`, text-only, 32K context, zero data retention). No weights, no disclosed parameter count, no license — nothing to run locally. That puts a cloud dependency inside the "fast" layer: fine at 70–500 ms for tick-level judgment, but the real robot needs a rules fallback for offline.
- **Multimodal?** No — text/structured state only; TypeSafe's own launch post says images "not yet". Jev cannot watch GPTPet's frames. It *can* read everything the MCP tools already emit as JSON — objects and passages with stable ids (`get_current_view`), `list_destinations`, `get_nav_status`, the goal record, the last report — plus the §13/§14 encoder signals (novelty / surprise / watch scores) once they exist.
- **Does the LLM still set direction?** Structurally guaranteed: Jev *cannot* author goals or free text at all. It only answers typed questions someone else wrote. So Gemini writes the questions and criteria at tick time (the DoReMi "LLM writes the monitors" pattern, upgraded to "the LLM writes the questions"), Jev answers them between ticks, and a hit or low confidence escalates. Judgment without authority.

Where it fits: the missing middle layer. §13's encoders produce *signals* but cannot judge; Gemini judges but at seconds and dollars. Jev is judgment in between:

| layer | substrate | latency | role |
|---|---|---|---|
| percept (always on) | local encoders (§13) | ~ms, free | frames → novelty / watch / surprise / stuck signals |
| judgment (per decision point) | Jev | 0.07–0.5 s, ~free | state + signals → goal done? wake S2? which destination? + confidence |
| deliberation (on escalation + heartbeat) | Gemini | seconds | goals, plans, language, actually looking at the frame |

Each layer delegates up exactly when its kind of processing runs out: encoders emit signals they can't interpret; Jev emits confidence it can't justify; Gemini asks the owner. This is SOFAI's metacognitive arbiter (§1) with the confidence half bought off the shelf — and Jev's one-parallel-call shape means §14's whole escalation table compiles to a single request: `goal_done: Noul`, `wake_s2: Noul`, `next_destination: Choice(list_destinations)`, `interestingness: Score`, where `goal_done` implements the orchestration study's best hand-back trigger (success detection) over structured state.

Speculative later stage: Jev as the *acting* S1 — `Choice` over {MoveAhead, RotateLeft, RotateRight, set_nav_goal(dest)} from structured state at 1–2 Hz under Gemini's current goal (the Doom demo is exactly this shape). Gemini stays goal author; Jev just picks the next primitive. Only after the arbiter role proves calibrated.

Caveats, honestly:

- Two weeks old, early access, and every benchmark is vendor-reported. The Register's [skeptical read](https://www.theregister.com/ai-and-ml/2026/09/16/typesafe-ai-debuts-model-for-machines-that-plays-doom/5296711): "hallucination-free" only means type-safe — structured output "does not preclude the possibility of being incorrect."
- Calibration on GPTPet-shaped judgments is unproven → run the replay eval below before wiring anything.
- SEO lookalike "docs" domains are already squatting the launch (jevapi.dev, thejevai.com). Use typesafe.ai and the Cloudflare listing only; never send robot state to unofficial endpoints.

Eval idea (no mocks; fits the live tier): replay **logged real tick states** from past runs; ask Jev `goal_done` and `next_destination` for each; score against what Gemini actually decided (and against ground truth where the goal record settles it); check that low-confidence answers are the ones Gemini overturned. Costs cents, and it decides whether Jev earns the arbiter seat.

Readings:

- [ ] **TypeSafe launch post** — [Introducing System One models and Jev](https://typesafe.ai/blog/introducing-system-one-models-and-jev). The claims, the parallel sampler, RLCD, "not on images (yet…)".
- [ ] **The Register** (2026-09-16) — [TypeSafe AI debuts model for machines that plays Doom](https://www.theregister.com/ai-and-ml/2026/09/16/typesafe-ai-debuts-model-for-machines-that-plays-doom/5296711). Independent coverage + caveats.
- [ ] **Cloudflare Workers AI listing** — [typesafe/jev](https://developers.cloudflare.com/ai/models/typesafe/jev/). API schema (state + Noul/Choice/Score), context, pricing, zero retention.
- [ ] Explainers: [HatchWorks](https://hatchworks.com/blog/gen-ai/system-one-models-jev/), [DigitalOcean](https://www.digitalocean.com/resources/articles/what-is-jev), [DataCamp](https://www.datacamp.com/blog/system-one-models-jev).

## 13.2 Nimble (Bespoke Labs) on Ollama — the local Jev (chosen 2026-09-30)

Decision: nimble replaces Jev (§13.1) as GPTPet's S1 decider. Same System One API, runs on this Mac, open weights. Integration design in §14.1.

- **What**: an open Jev-style decision model — a Qwen3.5-9B LoRA fine-tune trained on contrastive pairs (one fact flipped per pair, so the model learns which evidence changes a decision). Ollama ≥ 0.35 serves it at `POST /v1/systemone` with Jev's schema (`state` + named `choice` / `noul` / `score` questions → per-option probabilities + confidence), so Jev cloud stays a base-URL swap. Limits: ≤ 64 questions per request, ≤ 255 options per choice, 8,192-token prompt. Text only. `ollama pull nimble` = 9.5 GB.
- **Accuracy** (vendor eval, 324 held-out examples): nimble 90.1%, Jev 1.13.0 93.2%, base Qwen3.5-9B 66.4%.
- **Latency**: Bespoke median 106 ms on an M5 Pro; Ollama quotes ~91 ms on an M5 Max. The dev Mac is an M4 Pro (24 GB), without the M5's GPU neural accelerators, and GPTPet's view payload is longer than a support ticket — expect a few hundred ms per call (estimate; measure). Together AI's tev1 (4B, 0.8B; experimental) serves the same endpoint if it's too slow.
- **Calibration — the caveat that shapes the design**: probabilities are a softmax over the offered options, not correctness rates. The original release was overconfident at T=1 (mean picked-answer probability 0.89, 73% correct) and got a fitted T=2.179; the 2026-09-24 checkpoint runs at T=1.0 unfitted; rating questions came out under-confident after fitting. → Escalation thresholds come from GPTPet's own shadow-mode data, never from the raw numbers. And softmax always picks *something*, so every choice carries an explicit escape option.
- **License**: the original checkpoint is Apache 2.0 (Ollama's page says Apache 2.0); the HF v3 weights are CC BY-NC 4.0. Fine for a personal project; recheck before anything commercial.
- **Setup gap on this Mac**: Ollama is 0.32.15 (the app at /Applications/Ollama.app) → update to ≥ 0.35 before `/v1/systemone` exists.

Readings:

- [ ] **Ollama**: [Ollama now supports Jev-style decision models](https://ollama.com/blog/ollama-now-supports-jev-style-decision-models), [library/nimble](https://ollama.com/library/nimble).
- [ ] **bespokelabsai/nimble** ([GitHub](https://github.com/bespokelabsai/nimble)) — README: eval, calibration history, latency table, limits.
- [ ] **Model cards**: [Bespoke-Nimble-9B](https://huggingface.co/bespokelabs/Bespoke-Nimble-9B), [v3](https://huggingface.co/bespokelabs/Bespoke-Nimble-9B-v3).

## 14. Design sketch: System 1 for the current brain

Added 2026-09-30; S1 decider settled the same day (nimble, §13.2). How the §13–§13.2 machinery slots into the existing code. The tick workflow (`brain.py`: goal_setter → update_goal_memory → executor) stays untouched; Gemini remains the only author of goals and success criteria.

Two S1 layers, built in this order:

1. **Decider at the model-call boundary** (§14.1, first). Uses only the structured state the MCP tools already return; no new perception.
2. **Subconscious perception loop** (§14.2, later). Local encoders watching frames and waking the tick early. Needed on the real robot (people and pets move); low value in the static AI2-THOR world.

### 14.1 S1 at the model-call boundary (nimble)

Both existing agents get a `before_model_callback` that answers a turn itself when the turn is mechanical (rules) or a judgment nimble is confident about, and returns None otherwise — handing that turn to Gemini with the full context, image included. ADK 2.9.2 (verified in `base_llm_flow.py`): a returned `LlmResponse` skips the model and flows through normal postprocessing — function calls execute (ids populated), text is validated against `output_schema` and saved to `output_key`; async callbacks are awaited. Same callback idiom as `ToolBudget` and the media callback.

| turn | today | with S1 |
|---|---|---|
| goal status (goal_setter) | Gemini | nimble `choice` when confidently `continue`; else Gemini |
| new goal + success criteria | Gemini | Gemini, always |
| first look | Gemini | rule: `get_current_view` |
| pick destination / turn | Gemini (sees the image) | nimble `choice` over the view's ids + `look_left` / `look_right` / `ask_gemini`; Gemini if unsure |
| poll nav (≤ 6) | Gemini, one turn per poll | rule: wait ~1 s, `get_nav_status` |
| criteria met + report | Gemini | nimble `noul` + templated four-line report |

Goal-setter pass-through rules (from `apply_decision`): no current goal or the rest stand-in; owner goals queued; `attempts + 1 >= max_goal_attempts` (a synthesized continue would auto-abandon and start `brain.default_goal`, a goal Gemini didn't write); any done/abandoned verdict (Gemini must author the next goal); margin under threshold. Net: Gemini is consulted when a goal starts or ends, when S1 is unsure, and at least every `max_goal_attempts` ticks.

Executor escalation (return None → Gemini takes the turn): `ask_gemini` chosen; margin under threshold; tool error; budget refusal; Ollama unreachable.

Guardrails keep working: `ToolBudget` caps apply to synthesized tool calls; S1 turns don't consume `max_llm_calls_per_tick` (ADK increments the counter after the short-circuit). Synthesized responses carry `custom_metadata` (`{"layer": "s1", "by": "rules" | "nimble", "p": ...}`) so `TickResult` and the portal can show the S1/S2 split per tick — the "how unconscious is the pet" dial.

Honest accounting: most of the executor's Gemini turns are mechanical (the first look plus up to six polls). Rules remove those; nimble covers the two or three real judgments per tick.

Rollout: a `[decider]` ini section (`base_url`, `model = nimble`, `mode = off | shadow | gate`, `min_margin`). Shadow mode logs nimble's answer beside Gemini's actual decision on every judgment turn; because nimble's probabilities aren't calibrated (§13.2), the threshold comes from that data (the margin above which nimble agrees with Gemini ≥ 95%); then `gate`. Jev cloud is a `base_url` swap; tev1 a `model` swap.

### 14.2 Subconscious perception loop (later)

An asyncio task inside `gpt-pet serve` beside `PetRuntime` — the spot `brain.py`'s docstring reserves.

Data paths (all already exist):

- Frames during a tick: subscribe to `FrameStore` (the executor's after-tool callback already lands camera/map/depth frames there).
- Frames during rests: S1's own `McpProxy` calling `get_current_view` at ~1 Hz (serialized client; the executor is idle while resting). Consider a raw-frame tool in ai2thor-mcp for this path.
- Escalation: `PetRuntime.request_tick()` cuts the rest short; evidence lands in session state via a runtime event (the `_note_truncation` pattern) under an `s1_events` key rendered into the goal-setter prompt.
- Direction: `GoalDecision` grows a `watch` field — SigLIP labels + thresholds + wake conditions. Gemini programs S1's attention (the DoReMi pattern: the LLM writes the monitors), then goes quiet.

Per-frame pipeline (local, fp16, MPS):

1. `z = encode(frame)` — SigLIP 2-B to start (it does double duty), DINOv3-S/B if dense features earn their keep.
2. novelty = distance of `z` to the visited-embedding bank (PatchCore-style coreset — v1's img2vec "seen before" cache done right). Inserting into the bank *is* habituation.
3. watch scores = cosine against the text embeddings of Gemini's current watch labels.
4. Stage 1+: surprise = prediction error of a tiny action-conditioned predictor; stuck = commanded MoveAhead but the embedding barely moved.

Escalation contract (S1 → S2), ordered by the survey's evidence:

| trigger | signal | note |
|---|---|---|
| watch_hit | SigLIP score over threshold | the cheap form of "success detection", the most robust hand-back trigger in [arXiv 2606.10267](https://arxiv.org/abs/2606.10267) |
| stuck / nav failure | no embedding motion under a motion command; nav status | LLM-Planner-style replan trigger |
| surprise | predictor error, z-scored against an EMA baseline | the validated VoE signal ([arXiv 2502.11831](https://arxiv.org/abs/2502.11831)); Stage 1 |
| novelty | bank distance over threshold | curiosity food; also biases S1's frontier scoring without waking S2 |
| heartbeat | existing `tick_delay_s` / `goal_delay_s` | the floor that guarantees S2 regains control; S1 may **shorten** rests, never lengthen them — "not unconscious all day" holds by construction |

Guardrails: per-trigger refractory periods and z-score thresholds against running baselines, so a busy scene habituates instead of spamming ticks. Calibration from the field: Hi Robot re-runs its high level every 1 s; OpenHelix measured performance to be surprisingly insensitive to the slow-model interval; the orchestration study suggests 4–8 s in-episode. A pet's "episodes" are minutes, so rest windows of 1–5 min with event escalations should dominate — the payoff over today's fixed cadence is fewer Gemini calls *and* faster reactions.

Delegation readings (extends §1's list; from the 2026-09-30 survey):

- [ ] **What Matters in Orchestrating Robot Policies** (2026). [arXiv 2606.10267](https://arxiv.org/abs/2606.10267). Systematic switching-mechanism study: success detection beats fixed frequency (4–8 s if fixed) and tolerates a noisy detector; cross-episode summaries help while in-episode history summarization is neutral-to-negative.
- [ ] **When Should a Robot Think? (RARRL)** (2026). [arXiv 2603.16673](https://arxiv.org/abs/2603.16673). RL-learned policy for whether/which/how-much LLM reasoning to invoke, evaluated on ALFRED/AI2-THOR — the learned endgame for the table above.
- [ ] **Hi Robot** re-invocation rule ([arXiv 2502.19417](https://arxiv.org/html/2502.19417v1)): re-run the high level every 1 s OR on user interaction — heartbeat + interjection in four words.
- [ ] **DoReMi** (2023). [arXiv 2307.00329](https://arxiv.org/abs/2307.00329). The LLM generates constraints while planning; a cheap monitor checks them; a violation recalls the LLM immediately. The watch-list pattern.
- [ ] **KnowNo** (2023). [arXiv 2307.01928](https://arxiv.org/abs/2307.01928) + **Introspective Planning** (2024). [arXiv 2402.06529](https://arxiv.org/abs/2402.06529). Conformal "ask for help" when the plan set is ambiguous — the S2→owner escalation, later.
- [ ] **Helix / Helix 02** ([figure.ai](https://www.figure.ai/news/helix), [Helix 02](https://www.figure.ai/news/helix-02)): S2 VLM 7–9 Hz → one latent vector → S1 80M @ 200 Hz (02 adds a 1 kHz S0 motor prior). **GR00T N1** ([arXiv 2503.14734](https://arxiv.org/abs/2503.14734)): 10 Hz VLM ↔ 120 Hz diffusion S1 via cross-attention. **Galaxea G0** ([arXiv 2509.00576](https://arxiv.org/abs/2509.00576)): 2 Hz language subtasks → 200 Hz control. Index: [awesome-dual-system-vla](https://github.com/OpenHelix-robot/awesome-dual-system-vla).

Staged plan:

- **Decider stage (first)** — §14.1: shadow → gate. Needs Ollama ≥ 0.35 and `ollama pull nimble`.
- **Stage 0 — no training.** SigLIP 2-B encoder + memory-bank novelty + watch scoring + refractory logic. Ships the §14.2 contract end to end; the cell below gates it on this Mac's encode latency.
- **Stage 1 — tiny predictor.** Log `(z_t, action, z_t+1)` transitions during normal running (tool results already stream through the media callback), train a <10M MLP/GRU head on frozen embeddings — the DINO-WM/PLDM/JEPA-WMs recipe. Adds surprise + stuck. Discrete actions mean rollout scoring is enumeration, no CEM.
- **Stage 2 — temporal module.** V-JEPA 2 ViT-L (or 2.1 ViT-B @384) over an 8–16-frame ring buffer at ~0.5 Hz for motion-aware surprise — per-frame encoders are blind to motion, and moving people/pets are exactly what a pet should notice. MLX if MPS chokes.
- **Stage 3 — lookahead (speculative).** k-step latent rollouts to pre-rank destinations between ticks; still only ranking options S2 asked about.

Live acceptance (no mocks; extends `tests/live/`):

- Decider: real ticks with nimble in gate mode — the first call is still `get_current_view`, tool caps hold, S1 turns are tagged, and every new goal comes from an untagged (Gemini) goal-setter turn.
- Habituation: drive two identical laps via `McpProxy` (no LLM anywhere); assert mean novelty on lap 2 ≪ lap 1 and zero escalations on lap 2.
- Escalation: while the pet rests, change the world (new object / different `AI2THOR_SCENE` view); assert `request_tick` fired and the next tick's state carries the `s1_events` record (structure, never wording).
- Latency gates: nimble ms/decision and encoder ms/frame on the dev Mac (cell below for the encoders).
- Metrics for §9's list: Gemini calls/hour, S1/S2 decision ratio, event→tick latency, false-wake rate/hour, novel-views-per-minute.

Resolves §12's "S1 model choice": judgment = rules + nimble at the model-call boundary; perception (later) = frozen local encoders + embedding memory (+ tiny trained predictor); deliberation = Gemini. Open questions: nimble's shadow-mode agreement and M4 Pro latency; a raw-frame MCP tool; whether depth frames should join the embedding (the depth-view gap); when the Stage-1 predictor has enough logged transitions to be worth training.

In [ ]:
# §14 Stage 0 gate: encoder latency + novelty-bank demo on this machine.
# Deps (once, into the notebook's env):  %pip install torch transformers pillow
# Weights download from the HF Hub on first run. Flip RUN flags for the heavier models.

import time

import numpy as np
import torch

RUN = {"siglip2": True, "dinov2": True, "vjepa2_vitl": False}  # vjepa2: ~1.2 GB download, 16-frame clip input
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "mps" else torch.float32
FRAMES = np.random.randint(0, 256, (16, 256, 256, 3), dtype=np.uint8)  # stand-ins for get_current_view
# Real frame instead, with `gpt-pet serve` running:
#   import io, urllib.request; from PIL import Image
#   FRAMES[0] = np.array(Image.open(io.BytesIO(urllib.request.urlopen(
#       "http://127.0.0.1:8080/api/frame.jpg").read())).convert("RGB").resize((256, 256)))


def bench(label, fn, n=20):
    for _ in range(3):
        fn()
    if DEVICE == "mps":
        torch.mps.synchronize()
    t0 = time.perf_counter()
    for _ in range(n):
        fn()
    if DEVICE == "mps":
        torch.mps.synchronize()
    ms = (time.perf_counter() - t0) / n * 1000
    print(f"{label:30s} {ms:7.1f} ms/call  on {DEVICE}")
    return ms


embeds = {}

if RUN["siglip2"]:
    from transformers import AutoModel, AutoProcessor

    name = "google/siglip2-base-patch16-224"
    sig_proc = AutoProcessor.from_pretrained(name)
    sig = AutoModel.from_pretrained(name, torch_dtype=DTYPE).to(DEVICE).eval()

    @torch.no_grad()
    def siglip_embed(frame):
        px = sig_proc(images=frame, return_tensors="pt").pixel_values.to(DEVICE, DTYPE)
        return torch.nn.functional.normalize(sig.get_image_features(pixel_values=px), dim=-1)

    bench("SigLIP2-B image embed", lambda: siglip_embed(FRAMES[0]))
    embeds["siglip2"] = siglip_embed

    # Watch-list scoring: Gemini writes the labels, S1 does the arithmetic per frame.
    with torch.no_grad():
        labels = ["a person", "a dog or cat", "an open doorway", "a television"]
        tok = sig_proc(text=labels, padding="max_length", return_tensors="pt").to(DEVICE)
        z_text = torch.nn.functional.normalize(sig.get_text_features(**tok), dim=-1)
        scores = (siglip_embed(FRAMES[0]) @ z_text.T).squeeze().float().tolist()
        print("watch scores:", {l: round(s, 3) for l, s in zip(labels, scores)})

if RUN["dinov2"]:
    from transformers import AutoImageProcessor, AutoModel

    # 21M params, Apache 2.0. DINOv3-S is stronger but license-gated:
    # swap in "facebook/dinov3-vits16-pretrain-lvd1689m" after accepting its terms on the Hub.
    name = "facebook/dinov2-small"
    dino_proc = AutoImageProcessor.from_pretrained(name)
    dino = AutoModel.from_pretrained(name, torch_dtype=DTYPE).to(DEVICE).eval()

    @torch.no_grad()
    def dino_embed(frame):
        px = dino_proc(images=frame, return_tensors="pt").pixel_values.to(DEVICE, DTYPE)
        return torch.nn.functional.normalize(dino(pixel_values=px).pooler_output, dim=-1)

    bench("DINOv2-S image embed", lambda: dino_embed(FRAMES[0]))
    embeds["dinov2"] = dino_embed

if RUN["vjepa2_vitl"]:
    from transformers import AutoModel, AutoVideoProcessor

    name = "facebook/vjepa2-vitl-fpc64-256"
    vproc = AutoVideoProcessor.from_pretrained(name)
    vjepa = AutoModel.from_pretrained(name, torch_dtype=DTYPE).to(DEVICE).eval()

    @torch.no_grad()
    def vjepa_encode(frames):  # (T, H, W, 3) uint8 -> patch tokens, encoder only
        clip = torch.from_numpy(frames).permute(0, 3, 1, 2)
        px = vproc(clip, return_tensors="pt")["pixel_values_videos"].to(DEVICE, DTYPE)
        return vjepa(pixel_values_videos=px, skip_predictor=True).last_hidden_state

    # If MPS runs out of memory, drop to FRAMES[:8] or set DEVICE = "cpu" for this one.
    bench("V-JEPA2 ViT-L 16-frame clip", lambda: vjepa_encode(FRAMES), n=5)


class NoveltyBank:
    """Seen-it-before memory: novelty = 1 - max cosine sim to the bank; insert when novel (habituation)."""

    def __init__(self, threshold=0.15, cap=5000):
        self.z = None
        self.threshold = threshold
        self.cap = cap

    def score_and_update(self, z):
        z = z.squeeze(0).float().cpu()
        if self.z is None:
            self.z = z[None]
            return 1.0
        novelty = float(1 - (self.z @ z).max())
        if novelty > self.threshold and len(self.z) < self.cap:
            self.z = torch.cat([self.z, z[None]])
        return novelty


if embeds:
    fn = embeds.get("dinov2") or embeds["siglip2"]
    bank = NoveltyBank()
    scores = [bank.score_and_update(fn(f)) for f in FRAMES[:8]] + [bank.score_and_update(fn(FRAMES[0]))]
    print("novelty over 8 frames + 1 repeat:", [round(s, 3) for s in scores])
    # The repeat should score ~0. Uniform noise is a weak stand-in (noise embeddings cluster);
    # the real check is two identical laps in AI2-THOR with lap-2 scores collapsing (§14 live tests).